## Research Question

**Does receiving a personalized campaign (TypeA) cause higher household spend compared to a non-personalized campaign (TypeB/TypeC), and does this effect vary by household demographic profile?**

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA causal_project;


## Study Population and Design Rationale
- 45% of household x campaign rows had concurrent overlapping campaigns, first exposure guarantees clean single treatment 
- TypeB and TypeC collapsed into "non-personalized", TypeC n=18 after demographic filter (insufficient for stable estimates)
- TypeB and TypeC share the same targeting mechanism (uniform coupon distribution) justifying collapse
- 41 never-treated households excluded, insufficient sample size for a clean control group

In [0]:
%sql
-- Find number of households that were in multiple campaigns simultaneously
WITH overlap AS ( SELECT
ct.household_key,
  ct.campaign        AS campaign_a,
  cd_a.description   AS type_a,
  cd_a.start_day     AS start_a,
  cd_a.end_day       AS end_a,
  ct2.campaign       AS campaign_b,
  cd_b.description   AS type_b,
  cd_b.start_day     AS start_b,
  cd_b.end_day       AS end_b
FROM bronze_campaigns ct
JOIN bronze_campaigns ct2
  ON ct.household_key = ct2.household_key
  AND ct.campaign < ct2.campaign        
JOIN bronze_campaign_desc cd_a ON ct.campaign  = cd_a.campaign
JOIN bronze_campaign_desc cd_b ON ct2.campaign = cd_b.campaign
WHERE cd_a.start_day <= cd_b.end_day   
  AND cd_b.start_day <= cd_a.end_day
ORDER BY ct.household_key)
SELECT COUNT(*) From overlap

In [0]:
%sql
-- Total campaigns
SELECT COUNT(*) AS total_hh_campaign_rows
FROM bronze_campaigns;

In [0]:
%sql
WITH overlaps AS (
  SELECT 
    ct.household_key,
    ct.campaign        AS campaign_a,
    cd_a.description   AS type_a,
    cd_a.start_day     AS start_a,
    cd_a.end_day       AS end_a,
    ct2.campaign       AS campaign_b,
    cd_b.description   AS type_b
  FROM bronze_campaigns ct
  JOIN bronze_campaigns ct2
    ON ct.household_key = ct2.household_key
    AND ct.campaign < ct2.campaign
  JOIN bronze_campaign_desc cd_a ON ct.campaign  = cd_a.campaign
  JOIN bronze_campaign_desc cd_b ON ct2.campaign = cd_b.campaign
  WHERE cd_a.start_day <= cd_b.end_day
    AND cd_b.start_day <= cd_a.end_day
)
SELECT
  COUNT(DISTINCT household_key)          AS households_with_overlap,
  COUNT(*)                               AS total_overlap_pairs,
  -- What type combinations overlap most?
  type_a,
  type_b,
  COUNT(*)                               AS overlap_count
FROM overlaps
GROUP BY type_a, type_b
ORDER BY overlap_count DESC;

In [0]:
%sql
WITH demo_households AS (
  SELECT DISTINCT household_key FROM bronze_demographics
),
first_campaign AS (
  SELECT
    ct.household_key,
    cd.description AS first_campaign_type,
    ROW_NUMBER() OVER (
      PARTITION BY ct.household_key
      ORDER BY cd.start_day
    ) AS rn
  FROM bronze_campaigns ct
  JOIN bronze_campaign_desc cd ON ct.campaign = cd.campaign
  INNER JOIN demo_households d ON ct.household_key = d.household_key
)
SELECT first_campaign_type, COUNT(*) AS num_households
FROM first_campaign
WHERE rn = 1
GROUP BY first_campaign_type
ORDER BY num_households DESC;

In [0]:
%sql
SELECT COUNT(*) AS never_received_campaign
FROM bronze_demographics
WHERE household_key NOT IN (
  SELECT DISTINCT household_key FROM bronze_campaigns
);

## Treatment Definition
- Treatment: First campaign received was TypeA (personalized - 16 coupons selected by prior purchase behavior)  
- Treatment = 0: First campaign received was TypeB or TypeC (non-personalized)

## Outcome
- Primary: Total household spend during campaign window
- Secondary: Number of shopping trips during campaign window